<a href="https://colab.research.google.com/github/highpingShrew/AI-fashion-look/blob/main/notebooks/01_vision_prompt_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Vision Prompt Testing

Исследовательский ноутбук для тестирования vision-модели.

Цели:
- отправить фотографию предмета одежды;
- получить структурированный JSON;
- проверить ответ через `AIRecognitionResult`;
- сохранить сырой ответ и результат валидации.

Этапы Fashion Engine и AI Stylist в этом ноутбуке не рассматриваются.

In [ ]:
!git clone https://github.com/highpingShrew/AI-fashion-look.git

import os
import sys

os.chdir("/content/AI-fashion-look")
sys.path.insert(0, os.getcwd())

!pip install -q -r requirements.txt

In [ ]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

print("Ключ найден:", api_key is not None)

Ключ найден: True


In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [ ]:
from google.colab import files

uploaded = files.upload()

image_filename = next(iter(uploaded))

print("Загружен файл:", image_filename)

In [ ]:
BASELINE_PROMPT = """
Ты выполняешь только функцию AI Vision в приложении AI Fashion Wardrobe.

Твоя задача — проанализировать фотографию одного предмета одежды и вернуть
структурированный результат распознавания.

ОБЯЗАТЕЛЬНЫЕ ПРАВИЛА

1. Анализируй только предмет одежды, который виден на изображении.
2. Не подбирай образы и не давай стилистических рекомендаций.
3. Не придумывай характеристики, которые нельзя надёжно определить по фотографии.
4. Если значение невозможно определить, используй null.
5. Для списочных полей используй пустой список [], если значения не определены.
6. Добавляй название поля в uncertain_fields, если:
   - характеристика видна неоднозначно;
   - возможны несколько интерпретаций;
   - часть вещи закрыта;
   - качество фотографии мешает уверенному определению.
7. В image_quality_issues перечисляй только реальные проблемы изображения.
8. Не определяй бренд, стоимость, состав ткани, размер и другие отсутствующие в схеме данные.
9. Не добавляй новые поля.
10. Не возвращай Markdown, пояснения, комментарии или текст вне JSON.
11. Ответ должен быть синтаксически корректным JSON.
12. Все названия полей и строковые значения записывай на английском языке.
13. Поле description должно содержать краткое нейтральное описание только видимых характеристик вещи.

ПРАВИЛА ПО ПОЛЯМ

- category: верхнеуровневая категория вещи:
  top, bottom, dress, outerwear или shoes.
- type: основной тип вещи, например jeans, shirt, dress или sneakers.
- subtype: более точный подтип, только если он уверенно различим.
- primary_color: основной визуально доминирующий цвет.
- secondary_colors: только заметные дополнительные цвета.
- pattern: видимый тип рисунка или solid для однотонной вещи.
- fit: предполагаемая посадка только тогда, когда её можно определить по форме вещи.
- silhouette: общий видимый силуэт.
- style_tags: только стилистические признаки, которые подтверждаются изображением.
- season: подходящая сезонность, определяемая по видимой конструкции вещи.
- formality: визуально определимый уровень формальности.
- warmth_level: предполагаемый уровень утепления только по видимым признакам.
- statement_level: насколько вещь визуально акцентная.
- description: одно короткое фактическое предложение.

Верни JSON строго следующей структуры:

{
  "schema_version": "1.0",
  "status": "success",
  "fashion_attributes": {
    "category": null,
    "type": null,
    "subtype": null,
    "primary_color": null,
    "secondary_colors": [],
    "pattern": null,
    "fit": null,
    "silhouette": null,
    "style_tags": [],
    "season": [],
    "formality": null,
    "warmth_level": null,
    "statement_level": null,
    "description": null
  },
  "uncertain_fields": [],
  "image_quality_issues": []
}
"""

print(BASELINE_PROMPT)


Ты выполняешь только функцию AI Vision в приложении AI Fashion Wardrobe.

Твоя задача — проанализировать фотографию одного предмета одежды и вернуть
структурированный результат распознавания.

ОБЯЗАТЕЛЬНЫЕ ПРАВИЛА

1. Анализируй только предмет одежды, который виден на изображении.
2. Не подбирай образы и не давай стилистических рекомендаций.
3. Не придумывай характеристики, которые нельзя надёжно определить по фотографии.
4. Если значение невозможно определить, используй null.
5. Для списочных полей используй пустой список [], если значения не определены.
6. Добавляй название поля в uncertain_fields, если:
   - характеристика видна неоднозначно;
   - возможны несколько интерпретаций;
   - часть вещи закрыта;
   - качество фотографии мешает уверенному определению.
7. В image_quality_issues перечисляй только реальные проблемы изображения.
8. Не определяй бренд, стоимость, состав ткани, размер и другие отсутствующие в схеме данные.
9. Не добавляй новые поля.
10. Не возвращай Markdown, по

In [ ]:
import base64
import mimetypes
from pathlib import Path

mime_type, _ = mimetypes.guess_type(image_filename)
mime_type = mime_type or "image/jpeg"

with open(image_filename, "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": BASELINE_PROMPT,
                },
                {
                    "type": "input_image",
                    "image_url": f"data:{mime_type};base64,{image_data}",
                },
            ],
        }
    ],
)

print(response.output_text)

{
  "schema_version": "1.0",
  "status": "success",
  "fashion_attributes": {
    "category": "bottom",
    "type": "jeans",
    "subtype": "wide_leg",
    "primary_color": "dark_blue",
    "secondary_colors": [],
    "pattern": "solid",
    "fit": "loose",
    "silhouette": "wide_leg",
    "style_tags": [],
    "season": ["spring", "autumn", "winter"],
    "formality": "casual",
    "warmth_level": "medium",
    "statement_level": "low",
    "description": "Dark blue wide-leg jeans with a loose fit and solid pattern."
  },
  "uncertain_fields": [],
  "image_quality_issues": []
}
